<a href="https://colab.research.google.com/github/Bharath-Vikram/ybifoundation/blob/main/ybifoundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================================
CONTENT-BASED MOVIE RECOMMENDATION SYSTEM
================================================================================================
DESCRIPTION:
    This program builds an intelligent, content-based movie recommendation system
    using Natural Language Processing (NLP) and Machine Learning techniques.
    Unlike collaborative filtering (which relies on historical user ratings), content-based
    filtering analyzes the internal attributes of an item to recommend similar items.

HOW IT WORKS:
    1. Feature Fusion: Selected descriptive attributes of each movie (Genres, Keywords,
       Taglines, Cast, and Directors) are aggregated into a single combined text string.
    2. Tokenization & Vectorization: The combined text data is processed using Term Frequency-
       Inverse Document Frequency (TF-IDF). This converts text strings into numerical vectors
       where word frequencies are weighted against their uniqueness across the dataset.
    3. Similarity Matrix: A Cosine Similarity matrix is calculated across all vector pairs.
       The similarity score between two non-zero vectors $A$ and $B$ is formally computed as:

           Cosine Similarity(A, B) = (A Â· B) / (||A|| * ||B||)

       This yields a metric ranging from 0 (completely dissimilar) to 1 (identical).
    4. Fuzzy Text Matching: The system takes a text input string from the user. Using sequence
       matcher algorithms (difflib), it locates the closest matching title in the database
       to handle typos, capitalization mismatches, or incomplete titles gracefully.
    5. Ranking: The system identifies the target movie index, extracts its corresponding row
       from the similarity matrix, sorts the scores in descending order, and displays the top
       10 most similar recommendations.

DATASET CONFIGURATION:
    Target Dataset: YBI Foundation Movies Recommendation Corpus
    URL: https://raw.githubusercontent.com/YBIFoundation/Dataset/refs/heads/main/Movies%20Recommendation.csv
================================================================================================
"""

# ==========================================
# STEP 1: IMPORT CORE DEPENDENCIES
# ==========================================
import pandas as pd
import numpy as np
import difflib  # Used for fuzzy string matching to handle user spelling errors
from sklearn.feature_extraction.text import TfidfVectorizer  # Converts text into structural numeric arrays
from sklearn.metrics.pairwise import cosine_similarity  # Computes the angular distance metrics between data vectors

# ==========================================
# STEP 2: DATA COLLECTION AND INITIALIZATION
# ==========================================
# Loading the specific dataset requested by the user
dataset_url = 'https://raw.githubusercontent.com/YBIFoundation/Dataset/refs/heads/main/Movies%20Recommendation.csv'

print("[INFO] Fetching and loading movie metadata corpus from the specified repository...")
try:
    df = pd.read_csv(dataset_url)
    print(f"[SUCCESS] Dataset parsed successfully. Shape: {df.shape[0]} movies, {df.shape[1]} features.\n")
except Exception as e:
    print(f"[ERROR] Failed to download dataset remotely: {e}")
    print("[FALLBACK] Please ensure you are connected to the internet or provide a valid local file path.")
    exit()

# Column Schema Mapping for the YBI Foundation Dataset:
title_field = 'Movie_Title'
genre_field = 'Movie_Genre'
keywords_field = 'Movie_Keywords'
tagline_field = 'Movie_Tagline'
cast_field = 'Movie_Cast'
director_field = 'Movie_Director'

# Defining structural text variables required to compute similarity
selected_features = [genre_field, keywords_field, tagline_field, cast_field, director_field]

# ==========================================
# STEP 3: DATA PREPROCESSING & CLEANING
# ==========================================
# Machine learning algorithms cannot parse raw 'NaN' values. We must replace structural missing
# strings with empty character spacing to allow clean document string concatenations.
print("[INFO] Imputing missing text records and handling structural null variables...")
for feature in selected_features:
    df[feature] = df[feature].fillna('')

# ==========================================
# STEP 4: FEATURE FUSION / AGGREGATION
# ==========================================
# Merging all distinct categorical metadata rows into a single textual representation corpus string per movie.
print("[INFO] Executing comprehensive feature fusion across textual metadata elements...")
combined_text_corpus = (
    df[genre_field] + ' ' +
    df[keywords_field] + ' ' +
    df[tagline_field] + ' ' +
    df[cast_field] + ' ' +
    df[director_field]
)

# ==========================================
# STEP 5: NUMERICAL VECTORIZATION (TF-IDF)
# ==========================================
# Instantiating the Term Frequency-Inverse Document Frequency vectorizer to normalize common words
# (e.g., 'the', 'a', 'action') and emphasize critical character attributes unique to individual profiles.
print("[INFO] Transforming combined natural language text corpus into numeric feature arrays...")
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
feature_vectors = tfidf_vectorizer.fit_transform(combined_text_corpus)

# ==========================================
# STEP 6: SIMILARITY MATRIX COMPUTATION
# ==========================================
# Generating a dense matrix reflecting the multidimensional spatial distance profiles between every movie entry.
print("[INFO] Generating structural Cosine Similarity matrix profiles...")
similarity_matrix = cosine_similarity(feature_vectors)
print("[SUCCESS] Analytical model execution setup is complete.\n")

# ==========================================
# STEP 7: INTERACTIVE USER QUERY PROCESSING
# ==========================================
# Prompting the active user to enter their target reference query title.
user_input = input("Enter your favorite movie name to get recommendations: ").strip()

# Transforming the dataframe movie column index into a flat list array for string verification evaluation.
all_movie_titles = df[title_field].tolist()

# Implementing fuzzy matching to correct syntax, incomplete titles, or typing variations.
matched_titles = difflib.get_close_matches(user_input, all_movie_titles, n=3, cutoff=0.3)

if not matched_titles:
    print("\n[ALERT] Unable to safely locate a corresponding movie title match in the current dataset.")
    print("Please double check the title syntax or try specifying a different movie.")
else:
    # Capturing the primary closest matching title confirmed by the sequence verification algorithm.
    best_match_title = matched_titles[0]
    print(f"\n[MATCH FOUND] Closest historical record lookup match identified: \"{best_match_title}\"")

    # Locating the precise numerical index row where the matching title data entry resides.
    target_movie_index = df[df[title_field] == best_match_title].index[0]

    # ==========================================
    # STEP 8: TOP 10 RECOMMENDATION EXTRACTION
    # ==========================================
    # Extracting the complete vector similarity mapping row corresponding to our target database index match.
    # 'enumerate' wraps pairs of (movie_index, similarity_score) to prevent index loss during sorting operations.
    raw_similarity_scores = list(enumerate(similarity_matrix[target_movie_index]))

    # Sorting the similarity scores list in descending order based on the second value of the tuple element (index 1).
    sorted_recommendations = sorted(raw_similarity_scores, key=lambda x: x[1], reverse=True)

    print("\n" + "="*60)
    print(f"TOP 10 MOVIE RECOMMENDATIONS SIMILAR TO: '{best_match_title.upper()}'")
    print("="*60)

    rank = 1
    for movie_tuple in sorted_recommendations:
        movie_idx = movie_tuple[0]
        score = movie_tuple[1]

        # Extract the title name string linked to the parsed movie loop index variable
        title_name = df.iloc[movie_idx][title_field]

        # Filter out the original input movie title itself to prevent circular suggestions
        if title_name.lower() != best_match_title.lower():
            # Formatting data display metrics nicely to present a clean output profile interface
            print(f"{rank:02d}. {title_name:<50} (Match Confidence Score: {score:.2%})")
            rank += 1

            # Break process exactly when 10 recommendations are provided to the end user
            if rank > 10:
                break
    print("="*60)